# Logistic Regression

Run from `notebooks/` after `prepare_features.py`. Fit on 2020?2022 and evaluate on 2023.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.data.preprocess import notebook_root
from src.models.data import load_baseline_inputs
from src.models.logistic_regression import fit_logistic_models, evaluate_logistic_models, save_logistic_results

root = notebook_root()

## Load prepared features

Check artifact hashes, split years, labels, and matching rows for core and expanded.

In [ ]:
inputs = load_baseline_inputs(root)
pd.DataFrame({c: {split: item[split][0].shape for split in ('train', 'validation')}
              for c, item in inputs.items()})

## Fit the models

Use multinomial Logistic Regression with L2 regularization, C=1, and no class weighting. Preprocessing was fitted on training only.

In [ ]:
models = fit_logistic_models(inputs)
{c: {"iterations": model.n_iter_.tolist(), "parameters": model.get_params()} for c, model in models.items()}

## Evaluate validation predictions

Compare the core and expanded configurations on the same validation crashes.

In [ ]:
results = evaluate_logistic_models(inputs, models)
pd.DataFrame([result['metrics'] for result in results.values()])

In [ ]:
results['core']['per_class']

In [ ]:
results['core']['confusion'].pivot(index='actual', columns='predicted', values='rows')

## Save results

Small reports go in `results/logistic_regression/`; fitted pipelines and crash-level predictions stay local in `data/processed/predictions/logistic_regression/`.

In [ ]:
save_logistic_results(root, inputs, models, results)